
This is a notebook that makes sure that the silver pipeline created clean tables.  This notebook runs checks against the Silver tables before analysis is done.  It looks for missing values in the columns that cannot be empty, codes that fall outside their allowed set, child records that point to a report that does not exist, duplicate keys, and row counts that drift from what you expect.

Every check returns one of three verdicts. Pass means the data is fine. Warn means something is off but expected, for example forty percent of ages that FAERS never records. Fail means a rule broke and needs improvement.

In [ ]:
%pip install pyspark==3.5.0 delta-spark==3.2.0 -q

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = "/content/drive/MyDrive/faers-data-pipeline"
sys.path.insert(0, PROJECT_ROOT)

from src.utils.spark_session import get_spark
spark = get_spark("FAERS-DQ")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

SILVER_DIR = os.path.join(PROJECT_ROOT, "data/silver")
GOLD_DIR = os.path.join(PROJECT_ROOT, "data/gold")

print(f"✓ Spark {spark.version} ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
Mounted at /content/drive
✓ Spark 3.5.0 ready


Read all seven Silver tables and print their row counts. 

In [ ]:
demo_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "demo"))
drug_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "drug"))
reac_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "reac"))
outc_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "outc"))
rpsr_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "rpsr"))
ther_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "ther"))
indi_df = spark.read.format("delta").load(os.path.join(SILVER_DIR, "indi"))

print(f"{'Table':<8} {'Rows':>12}")
print("-" * 22)
for name, df in [("DEMO", demo_df), ("DRUG", drug_df), ("REAC", reac_df),
                  ("OUTC", outc_df), ("RPSR", rpsr_df), ("THER", ther_df),
                  ("INDI", indi_df)]:
    print(f"{name:<8} {df.count():>12,}")

Table            Rows
----------------------
DEMO          767,774
DRUG        3,519,718
REAC        2,683,365
OUTC          557,664
RPSR           23,897
THER        1,050,991
INDI        2,239,116



This function runs all of the checks.  Each row names the check, the table it ran against, the metric it measured, and the result.  

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime


def run_faers_dq_checks(spark, demo_df, drug_df, reac_df, outc_df,
                         ther_df, indi_df, rpsr_df, run_label="2024_Q1Q2"):
    ts = datetime.now().isoformat()
    checks = []

    # 1. Null checks
    # Critical columns fail on any null. Expected null columns only warn.
    null_checks = [
        # (table_name, df, column, criticality)
        ("DEMO", demo_df, "primaryid", "critical"),
        ("DEMO", demo_df, "caseid", "critical"),
        ("DEMO", demo_df, "caseversion", "critical"),
        ("DEMO", demo_df, "sex", "expected"),
        ("DEMO", demo_df, "age", "expected"),       # ~40% null is normal
        ("DEMO", demo_df, "event_dt", "expected"),   # ~46% null is normal
        ("DEMO", demo_df, "reporter_country", "expected"),
        ("DRUG", drug_df, "primaryid", "critical"),
        ("DRUG", drug_df, "drugname", "critical"),
        ("DRUG", drug_df, "role_cod", "critical"),
        ("REAC", reac_df, "primaryid", "critical"),
        ("REAC", reac_df, "pt", "critical"),
    ]

    for tbl_name, df, col_name, criticality in null_checks:
        total = df.count()
        nulls = df.filter(F.col(col_name).isNull() | (F.col(col_name) == "")).count()
        pct = round(nulls / total * 100, 2) if total > 0 else 0

        if criticality == "critical":
            result = "PASS" if nulls == 0 else "FAIL"
        else:
            result = "PASS" if pct == 0 else ("WARN" if pct < 50 else "FAIL")

        checks.append((
            f"null_{col_name}", tbl_name,
            f"{nulls:,} nulls ({pct}%)", result, ts, run_label
        ))

    # 2. Domain validation
    # sex values
    invalid_sex = demo_df.filter(
        F.col("sex").isNotNull() &
        ~F.col("sex").isin(["M", "F", "UNK", "NS"])
    ).count()
    checks.append((
        "domain_sex", "DEMO",
        f"{invalid_sex:,} invalid values",
        "PASS" if invalid_sex == 0 else "FAIL",
        ts, run_label
    ))

    # role_cod values
    invalid_role = drug_df.filter(
        F.col("role_cod").isNotNull() &
        ~F.col("role_cod").isin(["PS", "SS", "C", "I"])
    ).count()
    checks.append((
        "domain_role_cod", "DRUG",
        f"{invalid_role:,} invalid values",
        "PASS" if invalid_role == 0 else "FAIL",
        ts, run_label
    ))

    # outc_cod values
    valid_outcomes = ["DE", "LT", "HO", "DS", "CA", "RI", "OT"]
    invalid_outc = outc_df.filter(
        F.col("outc_cod").isNotNull() &
        ~F.col("outc_cod").isin(valid_outcomes)
    ).count()
    checks.append((
        "domain_outc_cod", "OUTC",
        f"{invalid_outc:,} invalid values",
        "PASS" if invalid_outc == 0 else "FAIL",
        ts, run_label
    ))

    # age_in_years should sit between 0 and 120 after the Silver transform
    invalid_age = demo_df.filter(
        F.col("age_in_years").isNotNull() &
        ((F.col("age_in_years") < 0) | (F.col("age_in_years") > 120))
    ).count()
    checks.append((
        "range_age_in_years", "DEMO",
        f"{invalid_age:,} out of range (0-120)",
        "PASS" if invalid_age == 0 else "WARN",
        ts, run_label
    ))

    # 3. Referential integrity
    demo_ids = demo_df.select("primaryid").distinct()

    for child_name, child_df in [("DRUG", drug_df), ("REAC", reac_df),
                                   ("OUTC", outc_df), ("RPSR", rpsr_df),
                                   ("THER", ther_df), ("INDI", indi_df)]:
        orphans = child_df.select("primaryid").distinct() \
            .subtract(demo_ids).count()
        checks.append((
            f"ref_integrity_{child_name}", child_name,
            f"{orphans:,} orphan primaryids",
            "PASS" if orphans == 0 else "FAIL",
            ts, run_label
        ))

    # 4. Duplicate detection
    demo_total = demo_df.count()
    demo_unique = demo_df.select("primaryid").distinct().count()
    dupes = demo_total - demo_unique
    checks.append((
        "duplicate_primaryid", "DEMO",
        f"{dupes:,} duplicate primaryids",
        "PASS" if dupes == 0 else "FAIL",
        ts, run_label
    ))

    demo_case_total = demo_df.select("caseid").distinct().count()
    case_dupes = demo_total - demo_case_total
    checks.append((
        "duplicate_caseid", "DEMO",
        f"{case_dupes:,} duplicate caseids",
        "PASS" if case_dupes == 0 else "WARN",
        ts, run_label
    ))

    # 5. Row count tracking
    for tbl_name, df in [("DEMO", demo_df), ("DRUG", drug_df),
                          ("REAC", reac_df), ("OUTC", outc_df),
                          ("THER", ther_df), ("INDI", indi_df)]:
        count = df.count()
        checks.append((
            f"row_count", tbl_name,
            f"{count:,} rows",
            "PASS",
            ts, run_label
        ))

    # Build result DataFrame
    schema = ["check_name", "table_name", "metric", "result", "timestamp", "run_label"]
    return spark.createDataFrame(checks, schema=schema)

✓ run_faers_dq_checks() defined



This runs the checks against the current Silver tables and print the full report, then a short summary that counts how many checks passed, warned, or failed.

In [ ]:
dq_results = run_faers_dq_checks(
    spark, demo_df, drug_df, reac_df, outc_df,
    ther_df, indi_df, rpsr_df,
    run_label="2024_Q1Q2_post_merge"
)

# Display all results
print("=" * 70)
print("DATA QUALITY REPORT — Silver Layer (2024 Q1+Q2)")
print("=" * 70)

dq_results.orderBy("table_name", "check_name").show(50, truncate=False)

# Summary counts
print("Summary:")
dq_results.groupBy("result").count().orderBy("result").show()

DATA QUALITY REPORT — Silver Layer (2024 Q1+Q2)
+---------------------+----------+----------------------+------+--------------------------+--------------------+
|check_name           |table_name|metric                |result|timestamp                 |run_label           |
+---------------------+----------+----------------------+------+--------------------------+--------------------+
|domain_sex           |DEMO      |0 invalid values      |PASS  |2026-03-30T04:53:07.271322|2024_Q1Q2_post_merge|
|duplicate_caseid     |DEMO      |0 duplicate caseids   |PASS  |2026-03-30T04:53:07.271322|2024_Q1Q2_post_merge|
|duplicate_primaryid  |DEMO      |0 duplicate primaryids|PASS  |2026-03-30T04:53:07.271322|2024_Q1Q2_post_merge|
|null_age             |DEMO      |306,715 nulls (39.95%)|WARN  |2026-03-30T04:53:07.271322|2024_Q1Q2_post_merge|
|null_caseid          |DEMO      |0 nulls (0.0%)        |PASS  |2026-03-30T04:53:07.271322|2024_Q1Q2_post_merge|
|null_caseversion     |DEMO      |0 nulls (0.0%)

These results are saved to the Delta Table in the gold folder.  

In [ ]:
dq_log_path = os.path.join(GOLD_DIR, "data_quality_log")
os.makedirs(GOLD_DIR, exist_ok=True)

(dq_results.write
    .format("delta")
    .mode("append")
    .save(dq_log_path))

# Read back to confirm
dq_log = spark.read.format("delta").load(dq_log_path)
print(f"✓ DQ log saved: {dq_log.count()} check results")
print(f"  Location: {dq_log_path}")
print(f"  Run labels: {[row.run_label for row in dq_log.select('run_label').distinct().collect()]}")

✓ DQ log saved: 30 check results
  Location: /content/drive/MyDrive/faers-data-pipeline/data/gold/data_quality_log
  Run labels: ['2024_Q1Q2_post_merge']



We split the verdicts into Failures and then Warnings.  

In [ ]:
print("FAILURES (must fix):")
failures = dq_results.filter(F.col("result") == "FAIL")
if failures.count() == 0:
    print("  None — all critical checks passed! ✓\n")
else:
    failures.show(truncate=False)

print("WARNINGS (expected but worth tracking):")
warnings = dq_results.filter(F.col("result") == "WARN")
if warnings.count() == 0:
    print("  None\n")
else:
    warnings.show(truncate=False)

FAILURES (must fix):
+-------------+----------+----------------------+------+--------------------------+--------------------+
|check_name   |table_name|metric                |result|timestamp                 |run_label           |
+-------------+----------+----------------------+------+--------------------------+--------------------+
|null_event_dt|DEMO      |424,183 nulls (55.25%)|FAIL  |2026-03-30T04:53:07.271322|2024_Q1Q2_post_merge|
+-------------+----------+----------------------+------+--------------------------+--------------------+

WARNINGS (expected but worth tracking):
+----------+----------+----------------------+------+--------------------------+--------------------+
|check_name|table_name|metric                |result|timestamp                 |run_label           |
+----------+----------+----------------------+------+--------------------------+--------------------+
|null_age  |DEMO      |306,715 nulls (39.95%)|WARN  |2026-03-30T04:53:07.271322|2024_Q1Q2_post_merge|
+----

Let's take a closer look at how much is missing among the DEMO fields.  High null rates on age and event date are normal for FAERS, so it doesn't show a problem.  

In [ ]:
import pandas as pd

total = demo_df.count()
null_fields = ["primaryid", "caseid", "event_dt", "age", "age_in_years",
               "sex", "reporter_country", "occr_country", "wt"]

null_data = []
for col_name in null_fields:
    if col_name in demo_df.columns:
        nulls = demo_df.filter(F.col(col_name).isNull() | (F.col(col_name) == "")).count()
        pct = round(nulls / total * 100, 1)
        null_data.append({"field": col_name, "null_count": nulls, "null_pct": pct})

null_pdf = pd.DataFrame(null_data).sort_values("null_pct", ascending=False)
print("DEMO Null Rates:")
print(null_pdf.to_string(index=False))

DEMO Null Rates:
           field  null_count  null_pct
              wt      639436      83.3
        event_dt      424183      55.2
    age_in_years      306723      39.9
             age      306715      39.9
    occr_country       74662       9.7
       primaryid           0       0.0
          caseid           0       0.0
reporter_country           0       0.0
             sex           0       0.0
